# backward-fn-signature — ex1: write log_back with the canonical (grad_out, out, x) signature

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `backward-fn-signature`. Running the final beacon cell reports progress against the `Backprop: backward fn signature` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: backward fn signature` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`backward-fn-signature`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "backward-fn-signature"
DD_SUBTOPIC = "Backprop: backward fn signature"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Backward-fn signature — quick refresher

In a manual autograd, every forward op `f(x, y, ...) -> out` is paired with **one backward fn per input position**. The canonical signature is:

```python
def f_back<i>(grad_out, out, *args, **kwargs):
    """dL/dargs[i] given dL/dout, cached out, and the original args."""
    ...
```

- `grad_out` — the upstream gradient `dL/dout`, same shape as `out`.
- `out` — the cached forward output (so you don't recompute).
- `*args, **kwargs` — the original forward inputs (one of them is the input you're differentiating w.r.t.).

The fn returns `dL/dargs[i]`, **same shape as `args[i]`**. For elementwise ops the local Jacobian is diagonal so you just multiply `grad_out` by the elementwise derivative — no actual matrix is materialized.

### Exercise 1 — write log_back with the canonical (grad_out, out, x) signature

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the (grad_out, out, x) backward-fn convention by writing log_back, the per-element gradient of torch.log.
> Keywords: backward-fn, log, elementwise-derivative, signature
> ```

**KCs targeted:** `backward-fn-signature`, `chain-rule-elementwise`

Implement `log_back(grad_out, out, x)`. This is the simplest elementwise backward fn — the per-position warm-up for the whole BACK_FUNCS table.

**The math.** `out = log(x)` ⇒ `d(out)/d(x) = 1/x` elementwise. By the chain rule, `dL/dx = dL/dout * d(out)/d(x) = grad_out / x`.

**The signature.** All ARENA back fns share the shape `(grad_out, out, *args, **kwargs) -> grad_in`:

- `grad_out` — upstream gradient `dL/dout`, same shape as `out`.
- `out` — the cached forward result `log(x)` (you don't need it for   log_back, but it's part of the contract).
- `x` — the original input to `log`.

Return `dL/dx` with the **same shape and dtype as `x`**.

Inputs are `torch.Tensor` for this drill; no autograd, no grad tracking — just elementwise tensor arithmetic.

In [ ]:
def log_back(grad_out: Tensor, out: Tensor, x: Tensor) -> Tensor:
    # d(log x)/dx = 1/x; chain rule → grad_out / x.
    # We never read `out` — but it's part of the contract so every
    # back fn has the same call shape (uniform dispatch).
    return grad_out / x


<details><summary>Solution</summary>

```python
def log_back(grad_out: Tensor, out: Tensor, x: Tensor) -> Tensor:
    # d(log x)/dx = 1/x; chain rule → grad_out / x.
    # We never read `out` — but it's part of the contract so every
    # back fn has the same call shape (uniform dispatch).
    return grad_out / x
```

**Why `out` even though we don't use it.** Uniform calling convention. The reverse pass dispatches `back_fn(grad_out, node.array, *node.recipe.args)` for every node, regardless of which fn it is. `exp_back` will need `out` (since `d(exp x)/dx = exp(x) = out`); `log_back` doesn't. Keeping the signature uniform means one dispatcher, no special cases.

**Elementwise = diagonal Jacobian.** For elementwise ops, the Jacobian is diagonal so the chain rule reduces to elementwise multiplication. No matrix is materialized. This is why log/exp/relu/etc. backward fns are one-liners.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()